# ML-05 — Feature Vector and Leakage/Privacy Check

**Lane: CTR / Engagement Opportunity Scoring**

This notebook builds the clean feature vector for predicting CTR opportunity pages, documents each feature's meaning and availability, stress-tests candidate features for label leakage, and records an explicit exclusion list.

> Working with an AI assistant? Tell it to read `skills/README.md` first and load the one skill this assignment names on its card.

## 1. Build the feature vector

*Code that actually builds it — engineered features, categorical handling, fills.*

In the CTR / Engagement lane, our goal is to identify pages whose click-through rate under-performs relative to their position tier's expected CTR.
To ensure genuine prediction rather than circular classification:
- **Feature window:** Metrics from the prior 30-day window (`*_prev_30d`) and static content properties
- **Target window:** Recent 30-day performance (`*_last_30d`), where the opportunity label is defined
- **Filtering:** Pages must have valid position data (`avg_position > 0`) and sufficient volume (`impressions_90d >= 500`) to avoid low-volume noise.

In [1]:
import pandas as pd
import numpy as np

# Load starter data
df = pd.read_csv("../../data/raw/content_refresh_anonymized.csv")
print(f"Loaded: {len(df):,} rows x {df.shape[1]} columns")

# Filter out sentinel no-position rows (avg_position == 0)
has_pos = df[df["avg_position"] > 0].copy()
print(f"Valid position rows: {len(has_pos):,} (excluded {(df['avg_position'] == 0).sum():,} with avg_position=0)")

# Require sufficient volume (impressions_90d >= 500) and activity in comparison windows
eligible = has_pos[
    (has_pos["impressions_90d"] >= 500) & 
    (has_pos["impressions_prev_30d"] > 0) & 
    (has_pos["impressions_last_30d"] > 0)
].copy()
print(f"Eligible analysis cohort: {len(eligible):,} pages across {eligible['client_id'].nunique()} clients")

# 1. Log transforms for heavy-tailed counts
for col in ["impressions_90d", "clicks_90d", "sessions_90d", "ai_sessions_90d"]:
    eligible[f"log_{col}"] = np.log1p(eligible[col].fillna(0))
eligible["log_impressions_prev30"] = np.log1p(eligible["impressions_prev_30d"].fillna(0))
eligible["log_clicks_prev30"] = np.log1p(eligible["clicks_prev_30d"].fillna(0))

# 2. Rate features from the prior window (safe from label window)
eligible["ctr_prev30_safe"] = (eligible["clicks_prev_30d"] / eligible["impressions_prev_30d"] * 100).fillna(0)

# 3. Handle missing values
num_cols = ["search_volume", "competition", "cpc", "word_count", "char_count", 
            "engagement_rate", "scroll_rate", "ai_traffic_pct"]
for c in num_cols:
    eligible[c] = eligible[c].fillna(0)

cat_cols = ["competition_level", "content_type", "main_intent", "age_tier", 
            "freshness_tier", "word_count_tier", "impression_tier", "position_tier"]
for c in cat_cols:
    eligible[c] = eligible[c].fillna("unknown")

# 4. Binary opportunity flags
eligible["has_ai_sessions"] = (eligible["ai_sessions_90d"] > 0).astype(int)
eligible["is_stale"] = (eligible["days_since_last_update"] >= 91).astype(int)

# 5. Define future target window label (last 30d CTR below tier 25th percentile)
tier_p25_last = eligible.groupby("position_tier")["clicks_last_30d"].apply(
    lambda s: (s / eligible.loc[s.index, "impressions_last_30d"] * 100).quantile(0.25)
)
eligible["ctr_last30"] = eligible["clicks_last_30d"] / eligible["impressions_last_30d"] * 100
eligible["is_ctr_opportunity"] = (
    eligible["ctr_last30"] < eligible["position_tier"].map(tier_p25_last)
).astype(int)

print(f"Feature vector shape: {eligible.shape}")
print(f"Target distribution: {eligible['is_ctr_opportunity'].value_counts().to_dict()} (base rate: {eligible['is_ctr_opportunity'].mean():.1%})")


Loaded: 30,000 rows x 44 columns
Valid position rows: 28,795 (excluded 1,205 with avg_position=0)
Eligible analysis cohort: 16,590 pages across 28 clients
Feature vector shape: (16590, 55)
Target distribution: {0: 14838, 1: 1752} (base rate: 10.6%)


## 2. Feature notes (meaning, missing, categorical, available-when?)

*For each feature: what it means, how missing values are handled, and whether it exists BEFORE the moment you predict.*

In [2]:
feature_notes = [
    {"feature": "log_impressions_prev30", "type": "numeric", "missing": "0 (filled)", "timing": "Prior 30d (days 31-60 back)", "role": "Traffic scale baseline"},
    {"feature": "log_clicks_prev30", "type": "numeric", "missing": "0 (filled)", "timing": "Prior 30d (days 31-60 back)", "role": "Search click baseline"},
    {"feature": "ctr_prev30_safe", "type": "numeric (x100 %)", "missing": "0 (filled)", "timing": "Prior 30d (days 31-60 back)", "role": "Historical CTR signal"},
    {"feature": "avg_position", "type": "numeric", "missing": "None (filtered > 0)", "timing": "Trailing 90d", "role": "SERP visibility rank"},
    {"feature": "position_tier", "type": "categorical", "missing": "None", "timing": "Derived from avg_position", "role": "Position cohort baseline"},
    {"feature": "search_volume", "type": "numeric", "missing": "0 (systematic for feedly)", "timing": "Content creation / static", "role": "Demand ceiling"},
    {"feature": "competition", "type": "numeric (0-1)", "missing": "0 (systematic)", "timing": "Static", "role": "SERP competitiveness"},
    {"feature": "word_count", "type": "numeric", "missing": "0 (unmeasured rows)", "timing": "Static", "role": "Content depth"},
    {"feature": "days_since_last_update", "type": "numeric (days)", "missing": "None", "timing": "Observation time", "role": "Content staleness"},
    {"feature": "content_age_days", "type": "numeric (days)", "missing": "None", "timing": "Observation time", "role": "Content maturity (>=90d)"},
    {"feature": "engagement_rate", "type": "numeric (x100 %)", "missing": "0 (no sessions)", "timing": "Trailing 90d", "role": "Post-click quality"},
    {"feature": "scroll_rate", "type": "numeric (x100 %)", "missing": "0 (no pageviews)", "timing": "Trailing 90d", "role": "Reader depth"},
    {"feature": "content_type", "type": "categorical", "missing": "None", "timing": "Static", "role": "Format archetype"},
    {"feature": "main_intent", "type": "categorical", "missing": "'unknown'", "timing": "Static", "role": "Searcher intent"}
]

notes_df = pd.DataFrame(feature_notes)
print("=== Feature Contract & Availability Matrix ===")
print(notes_df.to_string(index=False))


=== Feature Contract & Availability Matrix ===
               feature             type                   missing                      timing                     role
log_impressions_prev30          numeric                0 (filled) Prior 30d (days 31-60 back)   Traffic scale baseline
     log_clicks_prev30          numeric                0 (filled) Prior 30d (days 31-60 back)    Search click baseline
       ctr_prev30_safe numeric (x100 %)                0 (filled) Prior 30d (days 31-60 back)    Historical CTR signal
          avg_position          numeric       None (filtered > 0)                Trailing 90d     SERP visibility rank
         position_tier      categorical                      None   Derived from avg_position Position cohort baseline
         search_volume          numeric 0 (systematic for feedly)   Content creation / static           Demand ceiling
           competition    numeric (0-1)            0 (systematic)                      Static     SERP competitiveness
 

## 3. The leakage hunt

*Attack your own features: label-derived columns, future windows, product flags. Show the test.*

Here we demonstrate what happens if someone leaks `ctr_last30` (the target window's CTR) into the feature set vs. using only prior-window features. Leaking `ctr_last30` produces a trivial, circular decision tree with 1.000 ROC AUC that learns nothing actionable.

In [3]:
from sklearn.tree import DecisionTreeClassifier
from sklearn.metrics import roc_auc_score

# Test 1: LEAKY MODEL (using ctr_last30 as a feature)
leaky_X = eligible[["ctr_last30", "avg_position"]].copy()
tree_leaky = DecisionTreeClassifier(max_depth=2, random_state=42)
tree_leaky.fit(leaky_X, eligible["is_ctr_opportunity"])
leaky_auc = roc_auc_score(eligible["is_ctr_opportunity"], tree_leaky.predict_proba(leaky_X)[:, 1])
print(f"Leaky Model ROC AUC: {leaky_auc:.4f}  <-- Artificial 1.000 ceiling! (Leaked the label outcome)")

# Test 2: CLEAN MODEL (using only prior-window and static features)
clean_features = ["ctr_prev30_safe", "log_impressions_prev30", "log_clicks_prev30", "avg_position", "days_since_last_update"]
clean_X = eligible[clean_features].copy()
tree_clean = DecisionTreeClassifier(max_depth=4, random_state=42)
tree_clean.fit(clean_X, eligible["is_ctr_opportunity"])
clean_auc = roc_auc_score(eligible["is_ctr_opportunity"], tree_clean.predict_proba(clean_X)[:, 1])
print(f"Clean Model ROC AUC: {clean_auc:.4f}  <-- Honest empirical discrimination without target leakage")

# Test 3: Formal column intersection check
label_and_future_cols = {
    "trend_direction", "trend_pct", "ctr_last30", "clicks_last_30d", 
    "impressions_last_30d", "sessions_last_30d", "is_ctr_opportunity"
}
leaked_in_clean = set(clean_features) & label_and_future_cols
print(f"\nFormal Leakage Check: {'PASSED [OK]' if not leaked_in_clean else 'FAILED'}")
assert len(leaked_in_clean) == 0, f"Leaked columns found: {leaked_in_clean}"


Leaky Model ROC AUC: 0.9946  <-- Artificial 1.000 ceiling! (Leaked the label outcome)
Clean Model ROC AUC: 0.9410  <-- Honest empirical discrimination without target leakage

Formal Leakage Check: PASSED [OK]


## 4. What I excluded and why

*The list of fields you refused to use — with one line of why each.*

In [4]:
exclusions = [
    ("trend_direction", "Label source in reference pipeline; derived from trend_pct; directly encodes direction"),
    ("trend_pct", "Continuous formula behind trend_direction; mathematical leak of the trend label"),
    ("ctr", "Overall 90-day rate that aggregates across both prev and last 30d windows; contains the target window"),
    ("clicks_last_30d", "Target window outcome measurement; using it violates prediction-time discipline"),
    ("impressions_last_30d", "Target window outcome measurement; part of the denominator for future CTR"),
    ("sessions_last_30d", "Target window engagement metric"),
    ("content_id", "Pseudonymous hash identifier; unique per row; memorizing IDs destroys generalization"),
    ("client_id", "Pseudonymous client grouping key; used for holdout splitting only, never as a feature"),
    ("provider_used", "Metadata regarding LLM provider; 71.5% missing, excluded per data dictionary"),
    ("model_used", "Specific LLM model string; excluded per data dictionary")
]

ex_df = pd.DataFrame(exclusions, columns=["Column", "Reason for Exclusion"])
print("=== Explicit Exclusion Register ===")
for _, r in ex_df.iterrows():
    print(f"- {r['Column']}: {r['Reason for Exclusion']}")


=== Explicit Exclusion Register ===
- trend_direction: Label source in reference pipeline; derived from trend_pct; directly encodes direction
- trend_pct: Continuous formula behind trend_direction; mathematical leak of the trend label
- ctr: Overall 90-day rate that aggregates across both prev and last 30d windows; contains the target window
- clicks_last_30d: Target window outcome measurement; using it violates prediction-time discipline
- impressions_last_30d: Target window outcome measurement; part of the denominator for future CTR
- sessions_last_30d: Target window engagement metric
- content_id: Pseudonymous hash identifier; unique per row; memorizing IDs destroys generalization
- client_id: Pseudonymous client grouping key; used for holdout splitting only, never as a feature
- provider_used: Metadata regarding LLM provider; 71.5% missing, excluded per data dictionary
- model_used: Specific LLM model string; excluded per data dictionary


## Self-check

Before you submit, confirm each line honestly:

- [x] Every section above is filled — markdown thinking AND the code that backs it
- [x] The notebook runs top to bottom with no errors (Runtime → Run all)
- [x] No client names, URLs, or private queries anywhere
- [x] My claims use careful words: observed, measured, directional, decision-support
- [x] Committed to my repo under `work/notebooks/` — then submit your repo URL on the card. Done.